# ResNet50 From Scratch

This notebook prepares an 80/20 train-test split for the Open Images subset (`Chicken`, `Egg (Food)`, `Balloon`) and trains a ResNet50 with randomly initialized weights.

## Notebook Setup

If you open this notebook in Colab or Kaggle, clone the repository first:

```python
!git clone https://github.com/Nuray-Visne/image_classification.git
%cd /content/image_classification  # Colab
# or
%cd /kaggle/working/image_classification  # Kaggle
```

In [ ]:
%pip install -q torch torchvision matplotlib scikit-learn pillow

In [ ]:
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
def detect_env() -> str:
    if Path('/kaggle').exists():
        return 'kaggle'
    if Path('/content').exists():
        return 'colab'
    return 'local'


ENV = detect_env()
REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

if ENV == 'kaggle':
    BASE_DATA_DIR = Path('/kaggle/working/data')
elif ENV == 'colab':
    BASE_DATA_DIR = Path('/content/data')
else:
    BASE_DATA_DIR = REPO_ROOT / 'data'

DATA_ROOT = BASE_DATA_DIR / 'openimages_subset' / 'classification'
SPLIT_ROOT = BASE_DATA_DIR / 'openimages_subset_split'
RESULTS_ROOT = REPO_ROOT / 'results'
TRAIN_DIR = SPLIT_ROOT / 'train'
TEST_DIR = SPLIT_ROOT / 'test'

BASE_DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Environment:', ENV)
print('Data root:', DATA_ROOT)
print('Results root:', RESULTS_ROOT)

In [ ]:
if not DATA_ROOT.exists():
    from download_images import create_classification_dataset_from_openimages

    dataset = create_classification_dataset_from_openimages(
        class_names=['Egg (Food)', 'Chicken', 'Balloon'],
        output_dir=str(DATA_ROOT),
        max_images_per_class=200,
    )
else:
    print('Using existing dataset at', DATA_ROOT)

In [ ]:
def create_train_test_split(source_dir: Path, split_root: Path, train_ratio: float = 0.8, seed: int = 42):
    rng = random.Random(seed)

    if split_root.exists():
        shutil.rmtree(split_root)

    train_dir = split_root / 'train'
    test_dir = split_root / 'test'
    summary = {}

    for class_dir in sorted([p for p in source_dir.iterdir() if p.is_dir()]):
        files = sorted([p for p in class_dir.iterdir() if p.is_file()])
        if not files:
            continue

        rng.shuffle(files)
        split_idx = int(len(files) * train_ratio)
        train_files = files[:split_idx]
        test_files = files[split_idx:]

        class_train_dir = train_dir / class_dir.name
        class_test_dir = test_dir / class_dir.name
        class_train_dir.mkdir(parents=True, exist_ok=True)
        class_test_dir.mkdir(parents=True, exist_ok=True)

        for src in train_files:
            shutil.copy2(src, class_train_dir / src.name)

        for src in test_files:
            shutil.copy2(src, class_test_dir / src.name)

        summary[class_dir.name] = {'train': len(train_files), 'test': len(test_files)}

    return train_dir, test_dir, summary


TRAIN_DIR, TEST_DIR, split_summary = create_train_test_split(DATA_ROOT, SPLIT_ROOT, train_ratio=0.8, seed=SEED)
split_summary

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print('Classes:', train_dataset.classes)
print('Train samples:', len(train_dataset))
print('Test samples:', len(test_dataset))

In [ ]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

sum(p.numel() for p in model.parameters())

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_examples = 0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        total_examples += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / total_examples
    avg_acc = accuracy_score(all_labels, all_preds)
    return avg_loss, avg_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)

            total_loss += loss.item() * images.size(0)
            total_examples += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / total_examples
    avg_acc = accuracy_score(all_labels, all_preds)
    return avg_loss, avg_acc, all_labels, all_preds

In [ ]:
history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': [],
}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
    )

In [ ]:
final_test_loss, final_test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)
print(f'Final test loss: {final_test_loss:.4f}')
print(f'Final test accuracy: {final_test_acc * 100:.2f}%')

In [ ]:
results_summary = pd.DataFrame([
    {
        'model': 'resnet50_scratch',
        'seed': SEED,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'num_epochs': NUM_EPOCHS,
        'classes': ', '.join(train_dataset.classes),
        'train_samples': len(train_dataset),
        'test_samples': len(test_dataset),
        'final_test_loss': final_test_loss,
        'final_test_accuracy': final_test_acc,
        'num_parameters': int(sum(p.numel() for p in model.parameters())),
    }
])

history_df = pd.DataFrame({
    'epoch': list(range(1, NUM_EPOCHS + 1)),
    'train_loss': history['train_loss'],
    'train_acc': history['train_acc'],
    'test_loss': history['test_loss'],
    'test_acc': history['test_acc'],
})

summary_path = RESULTS_ROOT / 'resnet50_scratch_summary.csv'
history_path = RESULTS_ROOT / 'resnet50_scratch_history.csv'

results_summary.to_csv(summary_path, index=False)
history_df.to_csv(history_path, index=False)

print('Saved:', summary_path)
print('Saved:', history_path)

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history['train_loss'], label='Train Loss')
plt.plot(epochs, history['test_loss'], label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over 10 Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history['train_acc'], label='Train Accuracy')
plt.plot(epochs, history['test_acc'], label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy over 10 Epochs')
plt.legend()

plt.tight_layout()
fig_path = RESULTS_ROOT / 'resnet50_scratch_curves.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()